# Laya vs Phi-4-mini on ProofWriter: accuracy vs reasoning depth

Standalone Colab notebook. It needs no repo checkout.

**Question:** the [BoolQ experiment](https://github.com/CJosh88/slms/tree/main/experiments/2026-09-laya-vs-phi4-boolq) asked whether Laya can *find* an answer in a passage. This one asks whether it can *chain* facts. Laya decides in a single forward pass with no scratchpad, so it should do well when the answer is stated (depth 0) or one rule away (depth 1), then fall off as the proof gets longer. Phi-4-mini, and especially Phi-4-mini reasoning step by step, should degrade more slowly. The headline is **accuracy plotted against proof depth**, not a single leaderboard number.

| Arm | Params | How it answers |
|---|---|---|
| Laya (zero-shot) | 421M | One `choice` question (True / False / Unknown), one forward pass |
| Laya (fine-tuned) | 421M | The same, after fine-tuning on ~3k ProofWriter *train* questions spread over depths 0–5 |
| Phi-4-mini (direct) | 3.8B | Chat prompt, "Answer with only True, False or Unknown", greedy decoding |
| Phi-4-mini (CoT) | 3.8B | Chat prompt, "Reason step by step…", then `Answer: …`, greedy, up to 512 new tokens |

**Dataset:** the official AI2 ProofWriter release (`proofwriter-dataset-V2020.12.3`), open-world (OWA) `depth-5` split. Only OWA has the third label, *Unknown*. Each question's proof depth is the release's own `QDep` field. It is never recomputed here. The test sample is stratified to the same number of questions for every (depth, label) cell, depths 0–5 × {True, False, Unknown}, so chance is exactly 1/3 at every depth.

**Runtime → Change runtime type → T4 GPU** before running. Every model output and the fine-tuned checkpoint are cached, so a rerun after a disconnect resumes where it stopped. Set `USE_DRIVE = True` to keep them on Google Drive. The whole notebook takes roughly 1.5 hours on a T4, and most of that is Phi-4-mini's step-by-step reasoning.

In [ ]:
%pip install -q "laya==0.3.21" "transformers==4.57.3" "accelerate==1.11.0" safetensors scikit-learn scipy statsmodels

In [ ]:
import gc, json, math, os, random, re, time, zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch, transformers, laya
from tqdm.auto import tqdm

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("laya", laya.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)
assert torch.cuda.is_available(), "Fine-tuning needs a GPU: Runtime -> Change runtime type -> T4 GPU"
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
# T4 (compute capability 7.5) has no native bf16, so use fp16 there.
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

## 1. Config

In [ ]:
DEPTHS = list(range(0, 6))    # QDep 0-5; the release also has a handful of Unknowns at QDep 6-8, which are left out
LABELS = ["True", "False", "Unknown"]
N_PER_CELL = 50               # test questions per (QDep, label) cell -> 6 x 3 x 50 = 900; None = every question

# Laya fine-tuning (the single-T4 port of the official recipe from the BoolQ fine-tuning experiment)
N_TRAIN_PER_CELL = 167        # from meta-train -> ~3,000 items, balanced over depth x label
N_CALIB_PER_CELL = 17         # from meta-dev   -> ~300 items, for temperature fitting and per-epoch accuracy
LAYA_ID = "convaiinnovations/laya"
EPOCHS = 3
MICRO_BATCH = 8
GRAD_ACCUM = 8                # effective batch = MICRO_BATCH * GRAD_ACCUM = 64
GROUP_SIZE = 4                # noisy samples per item for the policy-gradient baseline
LR_ENCODER = 2.5e-5
LR_HEAD = 1.0e-4
SIGMA_START, SIGMA_END = 0.4, 0.1

# Phi-4-mini
PHI_ID = "microsoft/Phi-4-mini-instruct"
PHI_BATCH = 8                 # lower this if you hit CUDA out-of-memory errors
COT_MAX_NEW_TOKENS = 512

# One definition of the three labels, shared by every model.
CRITERIA = {
    "True": "the statement follows from the facts and rules",
    "False": "the negation of the statement follows from the facts and rules",
    "Unknown": "neither the statement nor its negation can be proved from the facts and rules",
}
LAYA_INSTRUCTIONS = "Using only the facts and rules in the text, is this statement True, False or Unknown? Statement: {question}"
SYSTEM_PROMPT = (
    "You judge a statement against a set of facts and rules, using only what is given. "
    "Answer True if the statement follows from them, False if its negation follows from them, "
    "and Unknown if neither can be proved. Something that is not stated is not automatically false."
)
DIRECT_TEMPLATE = "Facts and rules:\n{theory}\n\nStatement: {question}\nAnswer with only True, False or Unknown."
COT_TEMPLATE = (
    "Facts and rules:\n{theory}\n\nStatement: {question}\n"
    "Reason step by step, citing the facts and rules you use. Then finish with a final line of the form "
    "\"Answer: True\", \"Answer: False\" or \"Answer: Unknown\"."
)

PW_URL = "https://aristo-data-public.s3.amazonaws.com/proofwriter/proofwriter-dataset-V2020.12.3.zip"
PW_SPLIT_DIR = "proofwriter-dataset-V2020.12.3/OWA/depth-5"

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/slms/2026-09-laya-vs-phi4-proofwriter")
else:
    BASE_DIR = Path(".")
CACHE_DIR = BASE_DIR / "cache"
FIG_DIR = BASE_DIR / "figures"
FT_DIR = BASE_DIR / "laya_proofwriter_ft"   # fine-tuned checkpoint (~850 MB, not committed)
for d in (CACHE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 2. Load ProofWriter (official AI2 release)

The release zip (214 MB) is the file [allenai.org/data/proofwriter](https://allenai.org/data/proofwriter) links to. The splits are read straight out of the zip. Each theory record holds several questions, and each question carries `answer` (`true`, `false` or `"Unknown"`) and `QDep`, the depth of the proof the dataset authors generated for it. `QDep` is used as-is.

In [ ]:
import urllib.request

PW_ZIP = CACHE_DIR / "proofwriter-dataset-V2020.12.3.zip"
if not PW_ZIP.exists():
    print("Downloading", PW_URL)
    urllib.request.urlretrieve(PW_URL, PW_ZIP)
print(f"{PW_ZIP.name}: {PW_ZIP.stat().st_size / 1e6:.0f} MB")


def to_label(answer):
    if answer is True:
        return "True"
    if answer is False:
        return "False"
    if answer == "Unknown":
        return "Unknown"
    raise ValueError(f"unexpected answer {answer!r}")


def load_split(split):
    rows = []
    with zipfile.ZipFile(PW_ZIP) as z, z.open(f"{PW_SPLIT_DIR}/meta-{split}.jsonl") as f:
        for line in f:
            rec = json.loads(line)
            for qid, q in rec["questions"].items():
                rows.append({"id": f"{rec['id']}:{qid}", "theory_id": rec["id"], "theory": rec["theory"],
                             "question": q["question"], "label": to_label(q["answer"]), "qdep": int(q["QDep"])})
    return pd.DataFrame(rows)


def cell_counts(d):
    return (d.groupby(["qdep", "label"]).size().unstack(fill_value=0)
             .reindex(index=DEPTHS, columns=LABELS, fill_value=0))


def stratified(d, per_cell, name):
    # The same number of questions for every (QDep, label) cell; stop if any cell is empty.
    d = d[d.qdep.isin(DEPTHS)]
    counts = cell_counts(d)
    missing = [(q, l) for q in DEPTHS for l in LABELS if counts.loc[q, l] == 0]
    if missing:
        raise ValueError(f"{name}: no questions at all for (QDep, label) {missing}")
    parts = []
    for (q, l), g in d.groupby(["qdep", "label"]):
        if per_cell is not None and len(g) < per_cell:
            print(f"  warning: {name} QDep={q} {l} has only {len(g)} questions (< {per_cell}); taking all")
        parts.append(g if per_cell is None else g.sample(n=min(per_cell, len(g)), random_state=SEED))
    out = pd.concat(parts).sort_values(["qdep", "label", "id"]).reset_index(drop=True)
    assert (cell_counts(out) > 0).all().all()
    return out


full = {s: load_split(s) for s in ["train", "dev", "test"]}
print("Questions in the release (QDep x label, test split):")
print(cell_counts(full["test"]))

test_df = stratified(full["test"], N_PER_CELL, "test")
train_df = stratified(full["train"], N_TRAIN_PER_CELL, "train")
calib_df = stratified(full["dev"], N_CALIB_PER_CELL, "dev")

# No theory may appear in more than one of train / calibration / test.
theories = {n: set(d.theory) for n, d in [("train", train_df), ("calib", calib_df), ("test", test_df)]}
assert not theories["train"] & theories["test"] and not theories["calib"] & theories["test"] \
    and not theories["train"] & theories["calib"], "theory overlap between splits"

print(f"\nTest sample: {len(test_df)} questions from {test_df.theory_id.nunique()} theories")
print(cell_counts(test_df))
print(f"Train: {len(train_df)} | calibration: {len(calib_df)}")
for _, r in test_df[test_df.qdep == 3].head(1).iterrows():
    print(f"\n[{r.id}] QDep={r.qdep} gold={r.label}\n{r.theory}\nStatement: {r.question}")

## 3. Laya setup and input lengths

Laya's English checkpoint has a 512-token window. The question head (the instructions plus the three options) takes up to 192 of those, and the theory fills the rest. Theories are **not** truncated by this notebook, because cutting a fact or rule can change the gold answer. Instead, any input that fills the whole window is flagged as `overflow` and reported separately.

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

model_dir = snapshot_download(LAYA_ID)
_fix_tokenizer_config(model_dir)
tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    base_cfg = json.load(f)
MAX_LEN, HEAD_MAX_LEN = base_cfg["max_len"], base_cfg["head_max_len"]
print({"max_len": MAX_LEN, "head_max_len": HEAD_MAX_LEN}, "| QTYPES", QTYPES)


def laya_question(question):
    # Public API form, for agent.predict
    return {"type": "choice", "instructions": LAYA_INSTRUCTIONS.format(question=question), "criteria": CRITERIA}


def internal_question(question):
    # Internal form, for build_sequence (same text as laya_question)
    return {"t": "choice", "ins": LAYA_INSTRUCTIONS.format(question=question), "crit": CRITERIA}


# Option order is the criteria order, which is also the target order for fine-tuning.
opts = render_options(internal_question("x"))
assert [o.split(":")[0] for o in opts] == LABELS, opts
LABEL_INDEX = {l: i for i, l in enumerate(LABELS)}

for d in (test_df, train_df, calib_df):
    lens = [len(build_sequence(tok, t, internal_question(q), MAX_LEN, HEAD_MAX_LEN)[0])
            for t, q in zip(d.theory, d.question)]
    d["input_tokens"] = lens
    d["overflow"] = d.input_tokens >= MAX_LEN
print(test_df.groupby("qdep").input_tokens.describe()[["mean", "50%", "max"]].round(0))
print(f"\nInputs that fill the {MAX_LEN}-token window (possible truncation): "
      f"test {test_df.overflow.sum()}, train {train_df.overflow.sum()}, calib {calib_df.overflow.sum()}")

fig, ax = plt.subplots(figsize=(7, 3.5))
for q in DEPTHS:
    ax.hist(test_df[test_df.qdep == q].input_tokens, bins=30, histtype="step", label=f"QDep {q}")
ax.axvline(MAX_LEN, color="crimson", ls="--", label=f"Laya window ({MAX_LEN})")
ax.set_xlabel("Laya input tokens (question head + theory)"); ax.set_ylabel("Questions"); ax.legend(fontsize=8)
ax.set_title("ProofWriter input lengths by depth (test sample)")
fig.savefig(FIG_DIR / "token_lengths.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Run Laya (zero-shot)

Each question is asked as one `choice` question whose three options are True, False and Unknown, each with the definition from `CRITERIA`. Laya returns a probability for each option. Questions are scored one at a time (batch size 1), which is Laya's realistic single-request latency.

In [ ]:
def load_cache(path):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as f:
        return {rec["id"]: rec for rec in map(json.loads, f)}


def sync():
    torch.cuda.synchronize()


def run_laya(agent, d, stem, desc):
    path, meta_path = CACHE_DIR / f"{stem}.jsonl", CACHE_DIR / f"{stem}_meta.json"
    done = load_cache(path)
    todo = d[~d.id.isin(done)]
    print(f"{desc}: {len(done)} cached, {len(todo)} to run")
    if len(todo):
        torch.cuda.reset_peak_memory_stats()
        with path.open("a", encoding="utf-8") as f:
            for r in tqdm(todo.itertuples(), total=len(todo), desc=desc):
                sync(); t0 = time.perf_counter()
                a = agent.predict(r.theory, {"q": laya_question(r.question)})["answers"]["q"]
                sync(); latency = time.perf_counter() - t0
                f.write(json.dumps({"id": r.id, "pred": a["choice"], "latency_s": latency,
                                    **{f"p_{l}": float(a["probabilities"][l]) for l in LABELS}}) + "\n")
        meta_path.write_text(json.dumps({"peak_mem_gb": torch.cuda.max_memory_allocated() / 1e9, "params_m": 421.3}))
    return json.loads(meta_path.read_text())


agent = laya.load(LAYA_ID, device=DEVICE)

# Smoke test on two hand-written theories: depth 0 (stated) and depth 2 (two rules chained).
smoke = [
    ("Anne is kind. Bob is not big.", "Anne is kind.", "True"),
    ("Anne is kind. If someone is kind then they are round. If someone is round then they are blue.",
     "Anne is blue.", "True"),
    ("Anne is kind. Bob is not big.", "Bob is big.", "False"),
    ("Anne is kind. Bob is not big.", "Anne is green.", "Unknown"),
]
for theory, question, gold in smoke:
    a = agent.predict(theory, {"q": laya_question(question)})["answers"]["q"]
    print(f"{question:15s} gold={gold:8s} pred={a['choice']:8s}", {k: round(v, 3) for k, v in a["probabilities"].items()})

laya_meta = run_laya(agent, test_df, "laya", "Laya zero-shot")
print(laya_meta)
del agent; gc.collect(); torch.cuda.empty_cache()

## 5. Fine-tune Laya on ProofWriter

This is the single-T4 port of Laya's [official RLCD recipe](https://github.com/NandhaKishorM/laya/blob/main/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb) used in the [BoolQ fine-tuning experiment](https://github.com/CJosh88/slms/tree/main/experiments/2026-09-laya-finetune-boolq), with the same hyperparameters. The one change is that the question type is `choice` with the same three options as above, instead of `noul`, so the target is one-hot over [True, False, Unknown].

Training uses ~3,000 questions from ProofWriter's **train** split, balanced over depth 0–5 × label. A ~300-question slice from the **dev** split tracks accuracy per epoch and fits the `choice` temperature. The test questions come from theories the model never sees. The question this answers is whether training on multi-hop examples flattens Laya's depth curve, or whether the fall-off persists. If it persists, the single forward pass is the limit.

In [ ]:
def build_item(theory, question, label):
    seq, markers = build_sequence(tok, theory, internal_question(question), MAX_LEN, HEAD_MAX_LEN)
    if len(markers) != len(LABELS) or len(seq) >= MAX_LEN:   # drop rather than train on a cut theory
        return None
    target = [0.0] * len(LABELS)
    target[LABEL_INDEX[label]] = 1.0
    return {"ids": seq, "markers": markers, "qtype": QTYPES["choice"], "target": target, "label": LABEL_INDEX[label]}


def build_items(d):
    items = [build_item(t, q, y) for t, q, y in zip(d.theory, d.question, d.label)]
    kept = [it for it in items if it is not None]
    print(f"  built {len(kept)} / {len(items)} items ({len(items) - len(kept)} dropped), "
          f"max length {max(len(it['ids']) for it in kept)} tokens")
    return kept


print("train:"); train_items = build_items(train_df)
print("calib:"); calib_items = build_items(calib_df)

In [ ]:
from safetensors.torch import load_file, save_file
from laya.common import build_model, clamp_temperature, proper_reward

K = len(LABELS)


def collate(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, :len(it["ids"])] = torch.tensor(it["ids"])
        att[i, :len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, :k] = torch.tensor(it["target"])
    return {"input_ids": ids, "attention_mask": att, "marker_pos": mpos, "marker_mask": mmask,
            "target": target, "qtype": torch.tensor([it["qtype"] for it in items])}


def forward(model, batch):
    with torch.autocast("cuda", dtype=torch.float16):
        logits, act = model(*(batch[k].to(DEVICE) for k in
                              ("input_ids", "attention_mask", "marker_pos", "marker_mask", "qtype")))
    return logits.float(), act


@torch.no_grad()
def raw_logits(model, items, bs=16):
    model.eval()
    out = []
    for i in range(0, len(items), bs):
        logits, _ = forward(model, collate(items[i:i + bs], tok.pad_token_id))
        out.append(logits[:, :K].cpu())
    model.train()
    return torch.cat(out)


def fit_temperature(z, target):
    # Single temperature minimising cross-entropy on held-out items (as in the official recipe).
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(target * torch.log_softmax(z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(log_t.exp())


ft_done = (FT_DIR / "rl_agent_config.json").exists()
history = json.loads((CACHE_DIR / "train_history.json").read_text()) if ft_done else []
if not ft_done:
    cfg = dict(base_cfg)
    cfg["gradient_checkpointing"] = True
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    model.load_state_dict(load_file(os.path.join(model_dir, "model.safetensors")), strict=True)
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(DEVICE).train()

    enc_params = [p for n, p in model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW([{"params": enc_params, "lr": LR_ENCODER},
                                   {"params": head_params, "lr": LR_HEAD}], weight_decay=0.01)
    total_updates = math.ceil(len(train_items) / (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_updates, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda")
    calib_y = torch.tensor([it["label"] for it in calib_items])
    torch.cuda.reset_peak_memory_stats()
    t_start = time.time()

    for epoch in range(EPOCHS):
        rng = random.Random(SEED + epoch)
        items = train_items[:]
        rng.shuffle(items)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * epoch / max(1, EPOCHS - 1)
        optimizer.zero_grad(set_to_none=True)
        losses, rewards = [], []
        n_micro = math.ceil(len(items) / MICRO_BATCH)
        for step, b in enumerate(tqdm(range(0, len(items), MICRO_BATCH), desc=f"epoch {epoch + 1}/{EPOCHS}")):
            batch = collate(items[b:b + MICRO_BATCH], tok.pad_token_id)
            logits, act = forward(model, batch)
            mask = batch["marker_mask"].to(DEVICE)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(DEVICE)
            qtype = batch["qtype"].to(DEVICE)

            # Zero-mean Gaussian exploration on the logits, scored with a proper scoring rule
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=DEVICE) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), qtype, mask, w_sph=0.75, w_rps=1.0)
                adv = (r - r.mean(0, keepdim=True)) / ((r - r.mean(0, keepdim=True)).std() + 1e-6)
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + loss_ce) / GRAD_ACCUM + 0.0 * act.sum()

            scaler.scale(loss).backward()
            if (step + 1) % GRAD_ACCUM == 0 or step + 1 == n_micro:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update(); scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            losses.append(loss_ce.item()); rewards.append(r.mean().item())

        zc = raw_logits(model, calib_items)
        calib_acc = (zc.argmax(-1) == calib_y).float().mean().item()
        history.append({"epoch": epoch + 1, "ce_loss": float(np.mean(losses)), "reward": float(np.mean(rewards)),
                        "calib_acc": calib_acc, "minutes": (time.time() - t_start) / 60})
        print(history[-1])

    train_minutes = (time.time() - t_start) / 60
    train_peak_gb = torch.cuda.max_memory_allocated() / 1e9

    # Temperature calibration on the held-out slice, then save in the layout laya.Agent loads.
    # The agent picks the `choice:3-5` bucket for a 3-option question, so the fit goes there.
    zc = raw_logits(model, calib_items)
    t_choice = clamp_temperature(fit_temperature(zc, torch.tensor([it["target"] for it in calib_items])))
    temps = list(base_cfg.get("temperature", [1.0, 1.0, 1.0]))
    temps[QTYPES["choice"]] = t_choice
    by_options = {**base_cfg.get("temperature_by_options", {}), "choice:3-5": t_choice}
    print(f"Fitted choice temperature: {t_choice:.3f}")

    FT_DIR.mkdir(parents=True, exist_ok=True)
    save_file({k: v.half().contiguous().cpu() for k, v in model.state_dict().items()},
              str(FT_DIR / "model.safetensors"))
    model.encoder.config.save_pretrained(FT_DIR / "encoder")
    tok.save_pretrained(FT_DIR / "tokenizer")
    save_cfg = {**base_cfg, "fine_tuned": True, "model_name": "laya-proofwriter", "temperature": temps,
                "temperature_by_options": by_options}
    (FT_DIR / "rl_agent_config.json").write_text(json.dumps(save_cfg, indent=2))
    (CACHE_DIR / "train_history.json").write_text(json.dumps(history))
    (CACHE_DIR / "train_meta.json").write_text(json.dumps(
        {"train_minutes": train_minutes, "train_peak_gb": train_peak_gb, "choice_temperature": t_choice,
         "n_train_items": len(train_items), "n_calib_items": len(calib_items), "epochs": EPOCHS}))
    del model, optimizer, scheduler, scaler
    gc.collect(); torch.cuda.empty_cache()

train_meta = json.loads((CACHE_DIR / "train_meta.json").read_text())
print(train_meta)
pd.DataFrame(history)

In [ ]:
ft_agent = laya.Agent(str(FT_DIR), device=DEVICE)
laya_ft_meta = run_laya(ft_agent, test_df, "laya_ft", "Laya fine-tuned")
print(laya_ft_meta)
del ft_agent; gc.collect(); torch.cuda.empty_cache()

## 6. Run Phi-4-mini: direct, then step by step

**Direct:** greedy decoding with `max_new_tokens=3`. The probability of each label comes from a softmax over the first answer token's logits, with each label's spelling variants pooled via log-sum-exp (the same approach as the BoolQ P(yes)). The decoded text gives the prediction. When the text doesn't start with a label, the prediction falls back to the most probable label.

**Step by step (CoT):** greedy decoding with up to `COT_MAX_NEW_TOKENS` new tokens. The last `Answer: …` in the output is the prediction. An output with no parseable answer, for example one cut off at the token limit, counts as wrong, and the parse rate is reported. Latency is the batch's wall time divided by the batch size, so the longest trace in a batch sets it.

In [ ]:
from transformers import AutoModelForCausalLM

phi = {}   # loaded lazily, only if something is left to generate


def get_phi():
    if not phi:
        ptok = AutoTokenizer.from_pretrained(PHI_ID)
        ptok.padding_side = "left"
        if ptok.pad_token is None:
            ptok.pad_token = ptok.eos_token
        model = AutoModelForCausalLM.from_pretrained(PHI_ID, dtype=DTYPE, device_map="auto").eval()
        eos = model.generation_config.eos_token_id
        stop_ids = set(eos if isinstance(eos, list) else [eos]) | {ptok.pad_token_id}
        phi.update(tok=ptok, model=model, stop_ids=stop_ids)
        torch.cuda.reset_peak_memory_stats()
    return phi


def prompts_for(ptok, batch, template):
    return [ptok.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": template.format(theory=t, question=q)}],
        tokenize=False, add_generation_prompt=True) for t, q in zip(batch.theory, batch.question)]


def first_token_ids(ptok, words):
    return sorted({ptok.encode(w, add_special_tokens=False)[0] for w in words})


def parse_direct(text):
    t = text.strip().lower().lstrip("*'\"").strip()
    return next((l for l in LABELS if t.startswith(l.lower())), None)


ANSWER_RE = re.compile(r"answer\s*(?:is)?\s*[:：]?\s*\**\s*(true|false|unknown)\b", re.IGNORECASE)


def parse_cot(text):
    found = ANSWER_RE.findall(text)
    return found[-1].capitalize() if found else None


# Longest theories first, so each batch holds similar lengths and a CUDA OOM shows up early.
by_len = test_df.assign(n=test_df.theory.str.len()).sort_values("n", ascending=False)

direct_path = CACHE_DIR / "phi4_direct.jsonl"
todo = by_len[~by_len.id.isin(load_cache(direct_path))]
print(f"Phi-4-mini direct: {len(test_df) - len(todo)} cached, {len(todo)} to run")
if len(todo):
    P = get_phi(); ptok, model = P["tok"], P["model"]
    label_ids = {l: first_token_ids(ptok, [l, " " + l, l.lower(), " " + l.lower(), l.upper()]) for l in LABELS}
    all_ids = [i for ids in label_ids.values() for i in ids]
    assert len(all_ids) == len(set(all_ids)), f"label first tokens overlap: {label_ids}"
    print("label token ids", label_ids)
    with direct_path.open("a", encoding="utf-8") as f:
        for start in tqdm(range(0, len(todo), PHI_BATCH), desc="Phi-4-mini direct"):
            batch = todo.iloc[start:start + PHI_BATCH]
            enc = ptok(prompts_for(ptok, batch, DIRECT_TEMPLATE), return_tensors="pt", padding=True).to(model.device)
            sync(); t0 = time.perf_counter()
            with torch.no_grad():
                out = model.generate(**enc, max_new_tokens=3, do_sample=False, output_scores=True,
                                     return_dict_in_generate=True, pad_token_id=ptok.pad_token_id)
            sync(); per = (time.perf_counter() - t0) / len(batch)
            logits = out.scores[0].float()
            probs = torch.softmax(torch.stack([torch.logsumexp(logits[:, label_ids[l]], -1) for l in LABELS], -1), -1)
            texts = ptok.batch_decode(out.sequences[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            for id_, t, p in zip(batch.id, texts, probs.tolist()):
                parsed = parse_direct(t)
                f.write(json.dumps({"id": id_, "output": t, "parsed": parsed is not None,
                                    "pred": parsed or LABELS[int(np.argmax(p))], "latency_s": per,
                                    **{f"p_{l}": v for l, v in zip(LABELS, p)}}) + "\n")

In [ ]:
cot_path = CACHE_DIR / "phi4_cot.jsonl"
todo = by_len[~by_len.id.isin(load_cache(cot_path))]
print(f"Phi-4-mini CoT: {len(test_df) - len(todo)} cached, {len(todo)} to run")
if len(todo):
    P = get_phi(); ptok, model, stop_ids = P["tok"], P["model"], P["stop_ids"]
    with cot_path.open("a", encoding="utf-8") as f:
        for start in tqdm(range(0, len(todo), PHI_BATCH), desc="Phi-4-mini CoT"):
            batch = todo.iloc[start:start + PHI_BATCH]
            enc = ptok(prompts_for(ptok, batch, COT_TEMPLATE), return_tensors="pt", padding=True).to(model.device)
            sync(); t0 = time.perf_counter()
            with torch.no_grad():
                seqs = model.generate(**enc, max_new_tokens=COT_MAX_NEW_TOKENS, do_sample=False,
                                      pad_token_id=ptok.pad_token_id)
            sync(); per = (time.perf_counter() - t0) / len(batch)
            gen = seqs[:, enc["input_ids"].shape[1]:].tolist()
            for id_, g in zip(batch.id, gen):
                n_out = next((i for i, t in enumerate(g) if t in stop_ids), len(g))
                text = ptok.decode(g[:n_out], skip_special_tokens=True)
                pred = parse_cot(text)
                f.write(json.dumps({"id": id_, "output": text, "parsed": pred is not None, "pred": pred,
                                    "output_tokens": n_out, "hit_limit": n_out >= COT_MAX_NEW_TOKENS,
                                    "latency_s": per}) + "\n")

phi_meta_path = CACHE_DIR / "phi4_meta.json"
if phi:
    phi_meta_path.write_text(json.dumps({"peak_mem_gb": torch.cuda.max_memory_allocated() / 1e9,
                                         "params_m": sum(p.numel() for p in phi["model"].parameters()) / 1e6}))
    phi.clear(); gc.collect(); torch.cuda.empty_cache()
phi_meta = json.loads(phi_meta_path.read_text())
print(phi_meta)

## 7. Results

In [ ]:
from sklearn.metrics import f1_score


def wilson(k, n, z=1.96):
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return centre - half, centre + half


MODELS = {
    "Laya (zero-shot)": ("laya", laya_meta),
    "Laya (fine-tuned)": ("laya_ft", laya_ft_meta),
    "Phi-4-mini (direct)": ("phi4_direct", phi_meta),
    "Phi-4-mini (CoT)": ("phi4_cot", phi_meta),
}
frames = []
for name, (stem, _) in MODELS.items():
    g = test_df[["id", "qdep", "label", "overflow"]].merge(
        pd.DataFrame(load_cache(CACHE_DIR / f"{stem}.jsonl").values()), on="id")
    g["model"] = name
    frames.append(g)
res = pd.concat(frames, ignore_index=True)
res["pred"] = res.pred.fillna("none")          # unparsed CoT answers
res["correct"] = res.pred == res.label
assert res.groupby("model").size().eq(len(test_df)).all(), res.groupby("model").size()
res.to_csv(BASE_DIR / "results.csv", index=False)

# Chance = majority class within each depth (exactly 1/3 when the sample is balanced).
chance = test_df.groupby("qdep").label.agg(lambda s: s.value_counts(normalize=True).max())

rows = []
for name, g in res.groupby("model", sort=False):
    for q, gq in g.groupby("qdep"):
        lo, hi = wilson(gq.correct.sum(), len(gq))
        rows.append({"model": name, "qdep": q, "label": "all", "n": len(gq), "accuracy": gq.correct.mean(),
                     "ci_low": lo, "ci_high": hi})
        for l, gl in gq.groupby("label"):
            lo, hi = wilson(gl.correct.sum(), len(gl))
            rows.append({"model": name, "qdep": q, "label": l, "n": len(gl), "accuracy": gl.correct.mean(),
                         "ci_low": lo, "ci_high": hi})
by_depth = pd.DataFrame(rows)
by_depth.to_csv(BASE_DIR / "by_depth.csv", index=False)
acc = by_depth[by_depth.label == "all"].pivot(index="qdep", columns="model", values="accuracy")[list(MODELS)]
acc["chance"] = chance
print("Accuracy by QDep")
display(acc.style.format("{:.1%}"))

rows = [{"model": "Chance (majority class)", "n": len(test_df), "accuracy": float(chance.mean())}]
for name, g in res.groupby("model", sort=False):
    lo, hi = wilson(g.correct.sum(), len(g))
    meta = MODELS[name][1]
    rows.append({
        "model": name, "n": len(g), "accuracy": g.correct.mean(), "ci_low": lo, "ci_high": hi,
        "acc_qdep_0_1": g[g.qdep <= 1].correct.mean(), "acc_qdep_4_5": g[g.qdep >= 4].correct.mean(),
        "macro_f1": f1_score(g.label, g.pred, labels=LABELS, average="macro"),
        "parse_rate": g.parsed.mean() if "parsed" in g and g.parsed.notna().all() else float("nan"),
        "mean_output_tokens": g.output_tokens.mean() if g.output_tokens.notna().any() else float("nan"),
        "ms_per_example": 1000 * g.latency_s.mean(), "params_m": meta["params_m"], "peak_mem_gb": meta["peak_mem_gb"],
    })
summary = pd.DataFrame(rows).set_index("model")
summary.to_csv(BASE_DIR / "summary.csv")
print(f"Inputs over Laya's window: {test_df.overflow.sum()} of {len(test_df)}")
print(f"CoT outputs cut off at {COT_MAX_NEW_TOKENS} tokens: {res[res.model == 'Phi-4-mini (CoT)'].hit_limit.mean():.1%}")
print(f"Laya fine-tuning: {train_meta['train_minutes']:.1f} min, peak {train_meta['train_peak_gb']:.1f} GB, "
      f"{train_meta['n_train_items']} items x {train_meta['epochs']} epochs")
summary.style.format({
    "accuracy": "{:.1%}", "ci_low": "{:.1%}", "ci_high": "{:.1%}", "acc_qdep_0_1": "{:.1%}", "acc_qdep_4_5": "{:.1%}",
    "macro_f1": "{:.3f}", "parse_rate": "{:.1%}", "mean_output_tokens": "{:.0f}", "ms_per_example": "{:.1f}",
    "params_m": "{:,.0f}", "peak_mem_gb": "{:.2f}",
}, na_rep="–")

### Does Laya degrade faster?

A logistic regression of correctness on `QDep` for each arm gives its slope: the change in log-odds of a correct answer per extra step of proof. A more negative slope means faster degradation. To test whether two slopes really differ, one model is fitted across all arms with a `QDep × arm` interaction. It uses GEE clustered on question, because every arm answers the same questions. The reference arm is Laya (zero-shot), so each interaction term is how much *less steeply* (positive) or *more steeply* (negative) that arm degrades than zero-shot Laya.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

rows = []
for name, g in res.groupby("model", sort=False):
    d = g.assign(y=g.correct.astype(int))
    try:
        fit = smf.logit("y ~ qdep", data=d).fit(disp=0)
        lo, hi = fit.conf_int().loc["qdep"]
        rows.append({"model": name, "slope_log_odds_per_depth": fit.params["qdep"], "ci_low": lo, "ci_high": hi,
                     "p": fit.pvalues["qdep"]})
    except Exception as e:   # e.g. perfect separation when an arm is right on every question
        rows.append({"model": name, "slope_log_odds_per_depth": float("nan"), "note": str(e)[:80]})
slopes = pd.DataFrame(rows).set_index("model")
display(slopes.style.format({"slope_log_odds_per_depth": "{:+.3f}", "ci_low": "{:+.3f}", "ci_high": "{:+.3f}",
                             "p": "{:.2g}"}, na_rep="–"))

d = res.assign(y=res.correct.astype(int), arm=pd.Categorical(res.model, categories=list(MODELS)))
gee = smf.gee("y ~ qdep * arm", groups="id", data=d, family=sm.families.Binomial(),
              cov_struct=sm.cov_struct.Exchangeable()).fit()
inter = pd.DataFrame({"slope_diff_vs_laya_zero_shot": gee.params, "p": gee.pvalues}).filter(like="qdep:", axis=0)
inter.index = [i.split("[T.")[1].rstrip("]") for i in inter.index]
inter.style.format({"slope_diff_vs_laya_zero_shot": "{:+.3f}", "p": "{:.2g}"})

### Paired comparison at each depth

These are exact McNemar tests on the questions where exactly one arm of the pair is right.

In [ ]:
from scipy.stats import binomtest

wide = res.pivot(index="id", columns="model", values="correct").astype(bool).join(test_df.set_index("id").qdep)
PAIRS = [("Laya (zero-shot)", "Phi-4-mini (direct)"), ("Laya (zero-shot)", "Phi-4-mini (CoT)"),
         ("Laya (fine-tuned)", "Phi-4-mini (CoT)"), ("Laya (zero-shot)", "Laya (fine-tuned)")]
rows = []
for a, b in PAIRS:
    for q, g in wide.groupby("qdep"):
        only_a, only_b = int((g[a] & ~g[b]).sum()), int((~g[a] & g[b]).sum())
        rows.append({"pair": f"{a} vs {b}", "qdep": q, "diff_pts": 100 * (g[a].mean() - g[b].mean()),
                     "only_a": only_a, "only_b": only_b,
                     "p": binomtest(only_a, only_a + only_b, 0.5).pvalue if only_a + only_b else 1.0})
mcnemar = pd.DataFrame(rows)
mcnemar.pivot(index="qdep", columns="pair", values="diff_pts").round(1)

In [ ]:
mcnemar.pivot(index="qdep", columns="pair", values="p").style.format("{:.2g}").map(
    lambda p: "font-weight: bold" if p < 0.05 else "")

## 8. Figures

In [ ]:
COLORS = {"Laya (zero-shot)": "#9ecae9", "Laya (fine-tuned)": "#4C78A8",
          "Phi-4-mini (direct)": "#F58518", "Phi-4-mini (CoT)": "#B6451A"}
MARKERS = {"Laya (zero-shot)": "o", "Laya (fine-tuned)": "s", "Phi-4-mini (direct)": "^", "Phi-4-mini (CoT)": "D"}
allrows = by_depth[by_depth.label == "all"]

fig, ax = plt.subplots(figsize=(8.5, 5))
for m in MODELS:
    g = allrows[allrows.model == m].sort_values("qdep")
    ax.plot(g.qdep, g.accuracy, marker=MARKERS[m], color=COLORS[m], lw=2, label=m)
    ax.fill_between(g.qdep, g.ci_low, g.ci_high, color=COLORS[m], alpha=0.15)
ax.plot(chance.index, chance.values, color="grey", ls="--", label="chance (1/3)")
ax.set_xticks(DEPTHS); ax.set_ylim(0, 1.02)
ax.set_xlabel("Proof depth (ProofWriter QDep)"); ax.set_ylabel("Accuracy")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.set_title(f"ProofWriter (OWA): accuracy vs reasoning depth\n"
             f"{N_PER_CELL if N_PER_CELL else 'all'} questions per depth x label, 95% Wilson bands")
ax.legend(loc="lower left", fontsize=9); ax.grid(alpha=0.3)
fig.savefig(FIG_DIR / "accuracy_vs_depth.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Confusion matrices at shallow vs deep depths: where do wrong answers go?
PRED_COLS = LABELS + (["none"] if (res.pred == "none").any() else [])
fig, axes = plt.subplots(2, len(MODELS), figsize=(4.2 * len(MODELS), 7.5))
for row, (title, sel) in enumerate([("QDep 0-1", res.qdep <= 1), ("QDep 4-5", res.qdep >= 4)]):
    for col, m in enumerate(MODELS):
        g = res[sel & (res.model == m)]
        cm = pd.crosstab(g.label, g.pred, normalize="index").reindex(index=LABELS, columns=PRED_COLS, fill_value=0)
        ax = axes[row, col]
        ax.imshow(cm.values, cmap="Blues", vmin=0, vmax=1)
        for i in range(len(LABELS)):
            for j in range(len(PRED_COLS)):
                v = cm.values[i, j]
                ax.text(j, i, f"{v:.0%}", ha="center", va="center", color="white" if v > 0.6 else "black", fontsize=9)
        ax.set_xticks(range(len(PRED_COLS)), PRED_COLS); ax.set_yticks(range(len(LABELS)), LABELS)
        ax.set_title(f"{m}\n{title}", fontsize=10)
        if col == 0: ax.set_ylabel("Gold")
        if row == 1: ax.set_xlabel("Predicted")
plt.tight_layout()
fig.savefig(FIG_DIR / "confusion_by_model.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(4.2 * len(MODELS), 3.4), sharey=True)
lab = by_depth[by_depth.label != "all"]
for ax, m in zip(axes, MODELS):
    h = lab[lab.model == m].pivot(index="label", columns="qdep", values="accuracy").reindex(index=LABELS, columns=DEPTHS)
    im = ax.imshow(h.values, cmap="RdYlGn", vmin=0, vmax=1)
    for i in range(len(LABELS)):
        for j in range(len(DEPTHS)):
            ax.text(j, i, f"{h.values[i, j]:.0%}", ha="center", va="center", fontsize=8)
    ax.set_xticks(range(len(DEPTHS)), DEPTHS); ax.set_yticks(range(len(LABELS)), LABELS)
    ax.set_xlabel("QDep"); ax.set_title(m, fontsize=10)
fig.colorbar(im, ax=axes, shrink=0.8, label="Accuracy")
fig.savefig(FIG_DIR / "accuracy_depth_label_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

ax = axes[0]
lat = res.groupby(["model", "qdep"]).latency_s.mean().mul(1000).unstack(0)[list(MODELS)]
for m in MODELS:
    ax.plot(lat.index, lat[m], marker=MARKERS[m], color=COLORS[m], label=m)
ax.set_yscale("log"); ax.set_xticks(DEPTHS)
ax.set_xlabel("QDep"); ax.set_ylabel("ms per question (log scale)"); ax.legend(fontsize=8)
ax.set_title("Latency vs depth (Laya batch 1, Phi batch 8)")

ax = axes[1]
cot = res[res.model == "Phi-4-mini (CoT)"]
ax.boxplot([cot[cot.qdep == q].output_tokens for q in DEPTHS], positions=DEPTHS, widths=0.6)
ax.axhline(COT_MAX_NEW_TOKENS, color="crimson", ls="--", label="token limit")
ax.set_xlabel("QDep"); ax.set_ylabel("Output tokens"); ax.legend()
ax.set_title("Phi-4-mini CoT reasoning length vs depth")

ax = axes[2]
h = pd.DataFrame(history)
if len(h):
    ax.plot(h.epoch, h.calib_acc, "o-", color=COLORS["Laya (fine-tuned)"], label="Laya held-out (dev) accuracy")
    ax.axhline(summary.loc["Laya (zero-shot)", "accuracy"], color=COLORS["Laya (zero-shot)"], ls=":",
               label="Laya zero-shot (test)")
    ax.axhline(summary.loc["Phi-4-mini (CoT)", "accuracy"], color=COLORS["Phi-4-mini (CoT)"], ls="--",
               label="Phi-4-mini CoT (test)")
    ax.set_xticks(h.epoch); ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy"); ax.legend(fontsize=8)
ax.set_title(f"Laya fine-tuning ({train_meta['train_minutes']:.0f} min on this GPU)")

plt.tight_layout()
fig.savefig(FIG_DIR / "cost_and_finetuning.png", dpi=150, bbox_inches="tight"); plt.show()

### Deep questions where the arms disagree

These are depth-4 and depth-5 questions that zero-shot Laya got wrong and Phi-4-mini CoT got right, shown with the CoT trace.

In [ ]:
by = res.set_index(["id", "model"])
deep = wide[(wide.qdep >= 4) & ~wide["Laya (zero-shot)"] & wide["Phi-4-mini (CoT)"]].index[:4]
tq = test_df.set_index("id")
for id_ in deep:
    r = tq.loc[id_]
    print("=" * 100)
    print(f"[{id_}] QDep={r.qdep}  GOLD: {r.label}\n{r.theory}\nStatement: {r.question}\n")
    for m in MODELS:
        print(f"  {'OK ' if by.loc[(id_, m)].correct else 'BAD'} {m:20s} -> {by.loc[(id_, m)].pred}")
    print("\n  CoT trace:\n  " + str(by.loc[(id_, "Phi-4-mini (CoT)")].output).replace("\n", "\n  ")[:1500])

## 9. Save results

Download the zip and commit `figures/`, `results.csv`, `by_depth.csv` and `summary.csv` to this experiment folder. The dataset zip and the fine-tuned checkpoint are left out.

In [ ]:
import shutil
bundle = Path("/tmp/laya_vs_phi4_proofwriter_results")
if bundle.exists():
    shutil.rmtree(bundle)
shutil.copytree(CACHE_DIR, bundle / "cache", ignore=shutil.ignore_patterns("*.zip"))
shutil.copytree(FIG_DIR, bundle / "figures")
for name in ["results.csv", "by_depth.csv", "summary.csv"]:
    shutil.copy(BASE_DIR / name, bundle / name)
shutil.make_archive("laya_vs_phi4_proofwriter_results", "zip", bundle)
try:
    from google.colab import files
    files.download("laya_vs_phi4_proofwriter_results.zip")
except ImportError:
    print("Not on Colab; results in", BASE_DIR.resolve())